# GNSS antennas: marker, ARP, L1/L2 phase centers, PCO and PCV

**A short introduction using the actual DYNG RINEX header and `igs20.atx`,
2024-01-01.** We will use the **same `dsoclasses` calls** as our GPS code
positioning notebook. No adjustment, atmospheric model or phase observation
processing is needed.

Our question: *Where does a measured GNSS range begin and end?* The ground
marker coordinates, antenna reference point and frequency-dependent signal
phase centers are **different points**.


## 1. Which point is which?

| Point / quantity | Definition | Where is it given? |
|:--|:--|:--|
| **Marker** | Stable surveyed station point whose coordinates we ultimately report | RINEX `APPROX POSITION XYZ`; precise coordinates may come from SINEX |
| **ARP** | Physical **antenna reference point** (not the phase center) | RINEX `ANTENNA: DELTA H/E/N` gives marker $\rightarrow$ ARP |
| **L1 APC** | Mean antenna **phase center** for GPS L1 | ANTEX `G01` PCO, measured relative to the ARP |
| **L2 APC** | Mean antenna **phase center** for GPS L2 | ANTEX `G02` PCO, also relative to the ARP |
| **PCV** | Additional **direction-dependent phase-center variation** | ANTEX per-frequency curve versus zenith angle; our code uses `NOAZI` |

The L1 and L2 mean phase centers need **not** coincide. Furthermore PCV
does **not** mean the antenna mechanically moves with satellite elevation:
it is a calibration of the direction-dependent effective receiving point.

**Names that are easy to confuse:** `G01`, `G02` identify *ANTEX frequencies*,
not satellites `G01`, `G02`. RINEX observations `C1C`, `C1W`, etc. all belong
to the L1 band, while `C2W`, `C2X`, etc. belong to L2. The RINEX third
character specifies the signal/tracking attribute, **not a new ARP**.


## 2. Read the *real* DYNG antenna metadata and calibration

The file paths match our other introductory notebooks. Here we need only
the RINEX header and ANTEX; later, one short example uses GPS SP3 too.
The library returns ANTEX PCO and PCV in **metres** (the file stores mm).


In [ ]:
from os.path import join
import numpy as np
import matplotlib.pyplot as plt

from dsoclasses.rinex.gnss.rinex import GnssRinex, fetch
from dsoclasses.gnss.atx import Atx
from dsoclasses.gnss import systems as gs
from dsoclasses.geodesy import transformations

data_path = "../../data"
rinex_path = join(data_path, "DYNG00GRC_R_20240010000_01D_30S_MO.rnx")
antex_path = join(data_path, "igs20.atx")
sp3_path = join(data_path, "COD0MGXFIN_20240010000_01D_05M_ORB.SP3")

rnx = GnssRinex(rinex_path)
antex = Atx(antex_path)
antenna = antex.get_noazi(rnx.antenna_type, ['G01', 'G02'])

marker_xyz = np.asarray(rnx.approx_cartesian(), dtype=float)
latitude, longitude, height_m = transformations.car2ell(*marker_xyz)
if not all(hasattr(rnx, name) for name in ('dh', 'de', 'dn')):
    raise ValueError('This example needs RINEX ANTENNA: DELTA H/E/N')

print('Marker:', rnx.marker_name, '| receiver antenna:', rnx.antenna_type)
print('Approximate marker XYZ (ECEF) [m]:', marker_xyz)
print(f'RINEX H / E / N [m]: {rnx.dh:.4f} / {rnx.de:.4f} / {rnx.dn:.4f}')
print('ANTEX G01 PCO [North, East, Up] [m]:', antenna.pco('G01'))
print('ANTEX G02 PCO [North, East, Up] [m]:', antenna.pco('G02'))


## 3. Marker $\rightarrow$ ARP: the RINEX eccentricity

For a fixed station, RINEX records the displacement in the order
**Height, East, North**. In local ENU order, that is $(E,N,U)=(d_E,d_N,d_H)$.
The transformation in `dsoclasses` maps **ENU to ECEF**:

$$
\boxed{\mathbf r_{ARP}=\mathbf r_{marker}
+\mathbf R_{ENU\to ECEF}
\begin{bmatrix}d_E\\d_N\\d_H\end{bmatrix}.}
$$

That is a **physical antenna installation offset**, normally dominated by
the antenna height. It does not depend on whether the satellite sends L1
or L2 and does not change as the satellite moves.


In [ ]:
enu_to_ecef = transformations.geodetic2lvlh(latitude, longitude)
marker_to_arp_enu = np.array([rnx.de, rnx.dn, rnx.dh], dtype=float)
marker_to_arp_ecef = enu_to_ecef @ marker_to_arp_enu
arp_xyz = marker_xyz + marker_to_arp_ecef

print('Marker -> ARP (ENU) [m]:', marker_to_arp_enu)
print('Marker -> ARP (ECEF) [m]:', marker_to_arp_ecef)
print('ARP XYZ (ECEF) [m]:', arp_xyz)
print('Physical marker-to-ARP distance [m]:',
      np.linalg.norm(marker_to_arp_ecef))

# ECEF/ENU transformation is orthogonal and must preserve vector length.
assert np.allclose(enu_to_ecef.T @ enu_to_ecef, np.eye(3), atol=1e-12)
assert np.isclose(np.linalg.norm(marker_to_arp_enu),
                  np.linalg.norm(marker_to_arp_ecef), atol=1e-9)


## 4. ARP $\rightarrow$ L1/L2 mean phase centers: ANTEX PCO

For a stationary, north-aligned GNSS antenna, `antenna.pco('G01')` and
`antenna.pco('G02')` return **North, East, Up** vectors in metres. Be careful:
this is **NEU**, not RINEX **H/E/N** or the ENU order expected by the
coordinate rotation.

$$
\mathbf r_{APC,f}=\mathbf r_{ARP}+
\mathbf R_{ENU\to ECEF}
\begin{bmatrix}PCO_{E,f}\\PCO_{N,f}\\PCO_{U,f}\end{bmatrix}.
$$

These are **mean** phase centers. A line-of-sight-dependent PCV still
needs to be applied; a single fixed APC cannot represent every direction.


In [ ]:
def neu_to_ecef(neu):
    """Receiver ANTEX NEU [m] -> ECEF vector [m]."""
    north, east, up = np.asarray(neu, dtype=float)
    return enu_to_ecef @ np.array([east, north, up])

pco_l1_neu = antenna.pco('G01')
pco_l2_neu = antenna.pco('G02')
apc_l1_xyz = arp_xyz + neu_to_ecef(pco_l1_neu)
apc_l2_xyz = arp_xyz + neu_to_ecef(pco_l2_neu)

print('ARP -> L1 APC (ECEF) [m]:', neu_to_ecef(pco_l1_neu))
print('ARP -> L2 APC (ECEF) [m]:', neu_to_ecef(pco_l2_neu))
print('L1 to L2 mean phase-center separation [mm]:',
      np.linalg.norm(apc_l2_xyz-apc_l1_xyz)*1e3)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))

# These are separate scales: marker eccentricity is commonly decimetres/m;
# phase-center offsets are normally millimetres/centimetres.
ax[0].quiver([0], [0], [rnx.de], [rnx.dh], angles='xy',
             scale_units='xy', scale=1, color='tab:blue')
ax[0].scatter([0, rnx.de], [0, rnx.dh], c=['black', 'tab:blue'], s=55)
ax[0].annotate('Marker', (0, 0), xytext=(7, -15), textcoords='offset points')
ax[0].annotate('ARP', (rnx.de, rnx.dh), xytext=(7, 5), textcoords='offset points')
ax[0].set(xlabel='Local East [m]', ylabel='Local Up [m]',
          title='Physical marker → ARP (E–U projection)')
ax[0].margins(.35)

for freq, pco, color in [('L1/G01', pco_l1_neu, 'tab:blue'),
                         ('L2/G02', pco_l2_neu, 'tab:orange')]:
    n, e, u = pco*1e3
    ax[1].scatter(e, u, s=90, label=f'{freq}: N={n:.1f} mm', c=color)
    ax[1].plot([0, e], [0, u], c=color, lw=1)
ax[1].scatter([0], [0], marker='+', s=110, color='black', label='ARP')
ax[1].set(xlabel='PCO East [mm]', ylabel='PCO Up [mm]',
          title='L1/L2 mean phase centers (E–U projection)')
ax[1].legend(fontsize=8)
for a in ax:
    a.grid(alpha=.3)
    a.set_aspect('equal', adjustable='datalim')
plt.tight_layout()
plt.show()


## 5. Line-of-sight projection, PCV, and the **correction sign**

At satellite azimuth $A$ (clockwise from north) and elevation $E$, the
unit vector from the receiver to the satellite, in **NEU**, is

$$\mathbf u_{NEU}=
\begin{bmatrix}\cos E\cos A\\\cos E\sin A\\\sin E\end{bmatrix}.$$

The PCO projected on that direction is
$\delta_{PCO,f}=\mathbf{PCO}_{NEU,f}\cdot\mathbf u_{NEU}$.
The `NOAZI` PCV curve depends on **zenith angle** $z=90^\circ-E$.

In the sign convention of our **code-positioning notebooks**, the amount
**added** to the measured range to refer it to the **ARP** is

$$\boxed{\delta_{ARP,f}=\underbrace{\mathbf{PCO}_f\cdot\mathbf u}_{PCO}
+\underbrace{PCV_f(E)}_{\text{direction-dependent}}.}$$

In `dsoclasses` this is exactly
`antenna.range_correction_m('G01', azimuth_rad, elevation_rad)`.

**Sanity check of the sign:** moving the receiving point **toward** the
satellite shortens the geometric path. To compare the measured APC range
with a geometric range computed to the *earlier ARP*, we **add** the
positive projected PCO. ANTEX's PCV sign follows the calibration convention.


In [ ]:
def los_neu(azimuth, elevation):
    return np.array([np.cos(elevation)*np.cos(azimuth),
                     np.cos(elevation)*np.sin(azimuth),
                     np.sin(elevation)])

azimuth = np.deg2rad(50.0)
elevation = np.deg2rad(30.0)
unit_neu = los_neu(azimuth, elevation)

print('Example geometry: azimuth 50°, elevation 30°')
for frequency in ('G01', 'G02'):
    projected_pco = float(np.dot(antenna.pco(frequency), unit_neu))
    pcv_m = antenna.pcv_hgt(frequency, elevation)
    correction = antenna.range_correction_m(frequency, azimuth, elevation)
    assert np.isclose(projected_pco+pcv_m, correction, atol=1e-12)
    print(f'{frequency}: PCO projection = {projected_pco*1e3:+8.2f} mm; '
          f'PCV = {pcv_m*1e3:+8.2f} mm; '
          f'total to add = {correction*1e3:+8.2f} mm')

# First-order range geometry using a synthetic satellite direction.
# r_ARP - r_APC should equal +projected PCO (without PCV).
satellite_xyz_demo = arp_xyz + (enu_to_ecef @ unit_neu[[1, 0, 2]])*23e6
r_arp = np.linalg.norm(satellite_xyz_demo - arp_xyz)
r_apc_l1 = np.linalg.norm(satellite_xyz_demo - apc_l1_xyz)
print('Geometric ARP range minus L1 APC range [mm]:',
      (r_arp-r_apc_l1)*1e3)
assert np.isclose(r_arp-r_apc_l1, np.dot(pco_l1_neu, unit_neu), atol=1e-6)


In [ ]:
# ANTEX PCV is a function of zenith angle; the API accepts ELEVATION radians.
elevation_deg = np.linspace(10, 90, 81)
azimuth_rad = np.deg2rad(50)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for freq, label in [('G01', 'L1 / G01'), ('G02', 'L2 / G02')]:
    pcv_curve = np.array([antenna.pcv_hgt(freq, np.deg2rad(deg))
                          for deg in elevation_deg])*1e3
    correction_curve = np.array([
        antenna.range_correction_m(freq, azimuth_rad, np.deg2rad(deg))
        for deg in elevation_deg])*1e3
    ax[0].plot(elevation_deg, pcv_curve, label=label)
    ax[1].plot(elevation_deg, correction_curve, label=label)

ax[0].set(xlabel='Elevation [degrees]', ylabel='PCV [mm]',
          title='Actual DYNG receiver-antenna NOAZI curves')
ax[1].set(xlabel='Elevation [degrees]', ylabel='PCO + PCV [mm]',
          title='Total receiver correction (azimuth fixed at 50°)')
for a in ax:
    a.grid(alpha=.3)
    a.legend()
plt.tight_layout()
plt.show()


## 6. L1 and L2 must be corrected **separately**

For GPS dual-frequency **code**, the usual first-order ionosphere-free
combination is

$$P_{IF}=\frac{f_1^2 P_1-f_2^2 P_2}{f_1^2-f_2^2}.$$

Because the antenna corrections differ by frequency, compute
$\delta_1$ and $\delta_2$ first, then combine them using the **same
coefficients**:

$$\boxed{P_{IF}^{ARP}=P_{IF}^{raw}
+\frac{f_1^2\delta_1-f_2^2\delta_2}{f_1^2-f_2^2}.}$$

Do **not** choose a single “L1/L2 reference point,” nor simply average
the L1/L2 PCOs. Also, this antenna model does **not** replace satellite
code-bias/OSB corrections: those represent a separate error source.


In [ ]:
f1_hz, f2_hz = gs.GPS_L1_FREQ*1e6, gs.GPS_L2_FREQ*1e6
denominator = f1_hz**2-f2_hz**2
coef1 = f1_hz**2/denominator
coef2 = -f2_hz**2/denominator

delta_l1 = antenna.range_correction_m('G01', azimuth, elevation)
delta_l2 = antenna.range_correction_m('G02', azimuth, elevation)
delta_if = coef1*delta_l1+coef2*delta_l2
pco_if_neu = coef1*pco_l1_neu+coef2*pco_l2_neu

print(f'IF coefficients: {coef1:.6f} (L1), {coef2:.6f} (L2)')
print(f'L1 / L2 corrections to ARP [mm]: '
      f'{1e3*delta_l1:+.2f} / {1e3*delta_l2:+.2f}')
print(f'IF antenna correction to ARP [mm]: {1e3*delta_if:+.2f}')
print('IF-combined mean PCO NEU [mm]:', 1e3*pco_if_neu)
assert np.isclose(coef1+coef2, 1)

# The IF-combined PCO alone is NOT the total IF correction:
# the two frequency-dependent PCV terms also need to be combined.
assert np.isclose(delta_if,
                  (np.dot(pco_if_neu, unit_neu)
                   +coef1*antenna.pcv_hgt('G01', elevation)
                   +coef2*antenna.pcv_hgt('G02', elevation)))


## 7. One **real** GPS code observation: apply the antenna corrections

Find the first usable **visible** GPS satellite with both L1 and L2 code
measurements. We obtain its line of sight from the real precise orbit;
only this one sample is needed. The following values are deliberately
**not** atmospheric-, clock- or bias-corrected: we isolate **only** the
receiver-antenna contribution and show it in metres.

The third character of the chosen RINEX observation (`C1C`, `C2W`, etc.)
does not change the mapping `L1 → G01`, `L2 → G02`.


In [ ]:
from dsoclasses.orbits.interpolator import Sp3Interpolator
from dsoclasses.gnss import algorithms as alg

intrp = Sp3Interpolator.from_sp3(
    sp3_path, ['G'], interval_in_sec=2400,
    min_data_pts=8, stencil_pts=8, itype='Barycentric')
rinex_scale = (rnx.time_sys or '').strip().upper()
sp3_scale = intrp.time_system().strip().upper()
if not rinex_scale or rinex_scale != sp3_scale:
    raise ValueError(f'RINEX/SP3 time scales differ: {rinex_scale}/{sp3_scale}')

example_obs = None
for block in rnx:
    t = block.t()
    for sat, obs in block.filter_satellite_system('gps', False):
        try:
            first, code1 = fetch(obs, 'C1W', 'C1C', 'C1X', 'C1L')
            second, code2 = fetch(obs, 'C2W', 'C2C', 'C2D', 'C2P',
                                  'C2X', 'C2S', 'C2L')
            xs, ys, zs, _, tau = alg.sat_at_emission_time(arp_xyz, t, intrp, sat)
            sat_xyz = np.array([xs, ys, zs])
            _, az, el = transformations.azele(sat_xyz, arp_xyz)
        except (LookupError, ValueError, KeyError, RuntimeError):
            continue  # Expected unavailable sample, code pair or orbit window.
        if el >= np.deg2rad(20):
            example_obs = (t, sat, code1, float(first['value']),
                           code2, float(second['value']), az, el)
            break
    if example_obs is not None:
        break

if example_obs is None:
    raise LookupError('No visible GPS L1/L2 code pair in this SP3 window')

t, sat, code1, p1, code2, p2, az, el = example_obs
d1 = antenna.range_correction_m('G01', az, el)
d2 = antenna.range_correction_m('G02', az, el)
raw_if = coef1*p1+coef2*p2
arp_if = coef1*(p1+d1)+coef2*(p2+d2)

print('Selected satellite:', sat, '| epoch:', t)
print('Code descriptors:', code1, code2)
print(f'Azimuth / elevation [deg]: {np.rad2deg(az):.2f} / '
      f'{np.rad2deg(el):.2f}')
print(f'Raw L1/L2 code [m]: {p1:.3f} / {p2:.3f}')
print(f'Receiver antenna correction G01/G02 [mm]: '
      f'{d1*1e3:+.2f} / {d2*1e3:+.2f}')
print(f'Raw IF code [m]: {raw_if:.4f}')
print(f'IF code referred to ARP [m]: {arp_if:.4f}')
print(f'Difference [mm]: {(arp_if-raw_if)*1e3:+.2f}')
assert np.isclose(arp_if-raw_if, coef1*d1+coef2*d2, atol=1e-8)


## 8. How it enters our **code positioning** equations

In the positioning notebook the estimated XYZ refers to the **marker**.
We build the receiving point in two distinct steps:

1. From **marker to ARP** using the *known RINEX H/E/N vector* (and any
   solid-Earth-tide displacement). This changes the position passed into
   the geometric range computation.
2. From **frequency-dependent APC measurement to ARP** using the ANTEX
   *PCO projection + NOAZI PCV*, **added** separately to L1 and L2 code
   before the IF combination.

If we instead estimated the ARP coordinates directly, we would use a
different coordinate convention and need to convert back to the marker
before comparing to SINEX. **Never apply the marker-to-ARP offset twice.**

**Simplifications of this course:** `dsoclasses` uses the azimuth-independent
(`NOAZI`) receiver calibration; the antenna is assumed normally installed
and north-aligned. ANTEX describes antenna **phase-center** calibration;
its use in our code-only demonstration follows the existing positioning
model and does not replace signal-specific code-bias calibration. Satellite
antenna offsets are **a different problem**: SP3 satellite CoM and body-frame
transmitter PCO are treated separately by
`Atx.satellite_nadir_range_correction_m()` in the positioning exercise.

**Further reading** (official sources):

- [RINEX 3.05](https://files.igs.org/pub/data/format/rinex305.pdf),
  section 5.2.4 (marker, ARP, antenna eccentricity).
- [ANTEX format](https://files.igs.org/pub/resource/working_groups/antenna/ANTEX20.pdf)
  (reference points, PCO, PCV and units).
- [IGS: how to use antenna phase-center corrections](https://files.igs.org/pub/resource/working_groups/antenna/igs_antenna_pcc_gpsworld_2010.pdf).
- See our longer [`gps_code_model_analysis.ipynb`](gps_code_model_analysis.ipynb)
  for the complete code-observation model.
